# Toronto Warming Centre Capacity and Staffing Optimization

This notebook rebuilds the project as a standalone end-to-end analysis. The workflow moves from demand assumptions to simulation, staffing decisions, stress testing, capacity expansion, and sensitivity analysis.

The results should be read as scenario-based evidence rather than operational forecasts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
HOURS_OPEN = 12
EXTREME_PROB = 0.12
STAFF_RATIO = 20

CENTRES = pd.DataFrame({
    'centre': ['Elizabeth','George','Scarborough','North York','Spadina','Cecil','Jimmie Simpson'],
    'capacity': [75,30,68,46,22,30,30],
    'arrival_rate': [6.08,2.50,5.52,3.63,1.76,2.21,2.20],
    'extreme_only': [False,False,False,False,False,True,True]
})
CENTRES

## Demand simulation

In [ ]:
def simulate_demand(n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for scenario, mult in [('Moderate',1.0),('Extreme',1.10)]:
        for _, r in CENTRES.iterrows():
            lam = r.arrival_rate * HOURS_OPEN * mult
            if scenario == 'Moderate' and r.extreme_only:
                lam = 0
            for x in rng.poisson(lam, n):
                rows.append({'scenario':scenario,'centre':r.centre,'demand':int(x)})
    return pd.DataFrame(rows)

demand_sim = simulate_demand()
demand_sim.groupby(['scenario','centre'])['demand'].agg(['mean','std']).round(2)

## Staffing allocation

In [ ]:
MILP_STAFF = {
    'Elizabeth':4,'George':2,'Scarborough':4,'North York':3,
    'Spadina':2,'Cecil':2,'Jimmie Simpson':2
}

staff_check = CENTRES[['centre','capacity']].copy()
staff_check['staff'] = staff_check['centre'].map(MILP_STAFF)
staff_check['staff_supported_capacity'] = staff_check['staff'] * STAFF_RATIO
staff_check['effective_capacity'] = staff_check[['capacity','staff_supported_capacity']].min(axis=1)
staff_check

## 5,000-night stress test

In [ ]:
def evaluate_policy(capacity, staff, n=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    records=[]
    for night in range(n):
        extreme = rng.random() < EXTREME_PROB
        overflow=vacancy=served_local=total_demand=0
        for _, r in CENTRES.iterrows():
            if (not extreme) and r.extreme_only:
                demand=0
            else:
                mult=1.10 if extreme else 1.0
                demand=rng.poisson(r.arrival_rate*HOURS_OPEN*mult)
            eff=min(capacity[r.centre],staff[r.centre]*STAFF_RATIO)
            total_demand += demand
            served_local += min(demand,eff)
            overflow += max(demand-eff,0)
            vacancy += max(eff-demand,0)
        transferred=min(overflow,vacancy)
        turnaways=overflow-transferred
        served=served_local+transferred
        records.append({'night':night+1,'scenario':'Extreme' if extreme else 'Moderate',
                        'demand':total_demand,'served':served,'transferred':transferred,
                        'turnaways':turnaways,'service_level':served/total_demand if total_demand else 1.0})
    return pd.DataFrame(records)

base_caps = dict(zip(CENTRES.centre,CENTRES.capacity))
stress = evaluate_policy(base_caps,MILP_STAFF)
stress[['demand','turnaways','service_level']].describe(percentiles=[.5,.9,.95]).round(3)

## Interpretation

The staffing plan can support the available physical beds, so the analysis shifts from labour allocation to capacity allocation. This does not prove that staffing is irrelevant in practice; it means that, within the assumptions of this model, physical bed capacity becomes the more binding constraint.